In [ ]:
# Cell 1: Setup folders and paths
import os

# Base directory (root of the project)
BASE_PATH = os.path.join("..", "..")

# Path to your existing CSV splits (adjust if needed)
DATA_PATH = os.path.join(BASE_PATH, "datasets", "banglabook", "csv")

# Create folders for preprocessed data and results
PREPROCESSED_PATH = os.path.join(BASE_PATH, "preprocessed_dataset")
RESULTS_PATH = os.path.join(BASE_PATH, "results")

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print(f"Base path: {BASE_PATH}")
print(f"Data path: {DATA_PATH}")
print(f"Preprocessed path: {PREPROCESSED_PATH}")
print(f"Results path: {RESULTS_PATH}")
print("\nFolders ready.")

Base path: ..\..
Data path: ..\..\datasets\banglabook\csv
Preprocessed path: ..\..\preprocessed_dataset
Results path: ..\..\results

Folders ready.


In [19]:
# Cell 2: Load and examine the split data
import pandas as pd
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Load pre-split data
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
val = pd.read_csv(os.path.join(DATA_PATH, "validation.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train shape:", train.shape)
print("Val shape:", val.shape)
print("Test shape:", test.shape)
print("\nTrain columns:", train.columns.tolist())
print("\nFirst few rows of train:")
print(train.head(3))

Train shape: (110645, 9)
Val shape: (15806, 9)
Test shape: (31614, 9)

Train columns: ['id', 'Book_Name', 'Writer_Name', 'Category', 'Rating', 'Review', 'Site', 'sentiment', 'label']

First few rows of train:
      id                                          Book_Name  \
0  19635                নেভার স্টপ লার্নিং (হার্ডকভার)        
1  31528   রাসুলুল্লাহ (স) এর নামায (১ম ও ২য় খণ্ড একত্রে...   
2   9235                  পোয়েটিক জাস্টিস (পেপারব্যাক)        

                                   Writer_Name  \
0                                 আয়মান সাদিক    
1   আল্লামা মুহাম্মদ নাসীরুদ্দীন আলবানী (রহঃ)    
2                              আগাথা ক্রিস্টি    

                                            Category  Rating  \
0                                  ছাত্রজীবন উন্নয়ন        5   
1                                       সালাত/নামায        1   
2   রহস্য, গোয়েন্দা, ভৌতিক, মিথ, থ্রিলার, ও অ্যাড...       5   

                                              Review      Site sentiment  \
0  ম

In [20]:
# Cell 3: Check label distribution in splits
TEXT_COL = "Review"        # column containing the review text
LABEL_COL = "label"         # column with numeric labels (0,2)

def check_distribution(df, name):
    print(f"\n{name} distribution:")
    print(df[LABEL_COL].value_counts().sort_index())
    print(df[LABEL_COL].value_counts(normalize=True).sort_index().round(3))

check_distribution(train, "Train")
check_distribution(val, "Validation")
check_distribution(test, "Test")

# Check split ratio
total = len(train) + len(val) + len(test)
print(f"\nSplit ratios:")
print(f"Train: {len(train)/total:.2%}")
print(f"Val:   {len(val)/total:.2%}")
print(f"Test:  {len(test)/total:.2%}")


Train distribution:
label
0     6772
1     4763
2    99110
Name: count, dtype: int64
label
0    0.061
1    0.043
2    0.896
Name: proportion, dtype: float64

Validation distribution:
label
0      967
1      680
2    14159
Name: count, dtype: int64
label
0    0.061
1    0.043
2    0.896
Name: proportion, dtype: float64

Test distribution:
label
0     1935
1     1361
2    28318
Name: count, dtype: int64
label
0    0.061
1    0.043
2    0.896
Name: proportion, dtype: float64

Split ratios:
Train: 70.00%
Val:   10.00%
Test:  20.00%


In [21]:
# Cell 4: Convert to binary labels (0=Negative, 1=Positive)
def prepare_data(df):
    """Convert dataframe to binary format: text, label (0/1)"""
    df = df.copy()
    # Keep only needed columns and rename
    df = df[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label"})

    # Convert to string and drop empty
    df["text"] = df["text"].astype(str)
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.strip().astype(bool)]

    # Keep only 0 and 2, map 2→1, 0→0
    df = df[df["label"].isin([0, 2])]
    df["label"] = df["label"].map({0: 0, 2: 1}).astype(int)

    return df

# Prepare each split
train_bin = prepare_data(train)
val_bin = prepare_data(val)
test_bin = prepare_data(test)

print("Train binary shape:", train_bin.shape)
print("Label distribution in train:")
print(train_bin["label"].value_counts())
print("0 = Negative, 1 = Positive")

# Save binary versions in preprocessed folder
train_bin.to_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_train_binary.csv"), index=False)
val_bin.to_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_val_binary.csv"), index=False)
test_bin.to_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_test_binary.csv"), index=False)

print(f"\nBinary splits saved to: {PREPROCESSED_PATH}")

Train binary shape: (105882, 2)
Label distribution in train:
label
1    99110
0     6772
Name: count, dtype: int64
0 = Negative, 1 = Positive

Binary splits saved to: ..\..\preprocessed_dataset


In [ ]:
# Cell 5: Zero-shot evaluation with XLM-RoBERTa (honest: neutrals count as wrong)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

# Load binary test set
test_df = pd.read_csv(os.path.join(PREPROCESSED_PATH, "BanglaBook_test_binary.csv"))
test_df = test_df.rename(columns={"text": "text", "label": "label"})
test_df["text"] = test_df["text"].astype(str)
test_df["label"] = test_df["label"].astype(int)

# Create HF dataset
test_ds = Dataset.from_pandas(test_df[["text", "label"]])

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

test_ds = test_ds.map(tok, batched=True)
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model (3-class: neg/neu/pos)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Predict
trainer = Trainer(model=model, data_collator=collator)
pred = trainer.predict(test_ds)

logits = pred.predictions
pred_3class = np.argmax(logits, axis=1)  # 0=neg, 1=neu, 2=pos

# Honest mapping: neutral → wrong answer
y_true = test_df["label"].values
pred_mapped = np.where(pred_3class == 2, 1, 0)          # pos(2)→1, neg(0)→0
pred_honest = pred_mapped.copy()
neutral_mask = (pred_3class == 1)
pred_honest[neutral_mask] = 1 - y_true[neutral_mask]   # force opposite of true

# Metrics
acc = accuracy_score(y_true, pred_honest)
f1 = f1_score(y_true, pred_honest)
cm = confusion_matrix(y_true, pred_honest)

# Neutral statistics
neutral_on_neg = np.sum((y_true == 0) & (pred_3class == 1))
neutral_on_pos = np.sum((y_true == 1) & (pred_3class == 1))
total_neutral = np.sum(neutral_mask)

print("="*60)
print("ZERO-SHOT EVALUATION - BanglaBook")
print("="*60)
print(f"Model: {MODEL_NAME}")
print(f"Test samples: {len(test_df)}")
print(f"\nAccuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"F1-score: {f1:.4f} ({f1*100:.2f}%)")
print("\nConfusion Matrix [ [TN FP], [FN TP] ]:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_true, pred_honest, target_names=["NEG", "POS"]))
print(f"\nNeutral predictions: {total_neutral} ({total_neutral/len(y_true)*100:.2f}%)")
print(f"  True Negative → Neutral: {neutral_on_neg}")
print(f"  True Positive → Neutral: {neutral_on_pos}")

# Save results
with open(os.path.join(RESULTS_PATH, "banglabook_zero_shot.txt"), "w", encoding="utf-8") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write("DATASET: BanglaBook (binary: 0=NEG, 1=POS)\n")
    f.write("EVALUATION: Honest approach (neutral predictions count as wrong)\n\n")
    f.write(f"Test samples: {len(test_df)}\n")
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"F1-score: {f1:.4f}\n\n")
    f.write("Confusion Matrix [ [TN FP], [FN TP] ]:\n")
    f.write(str(cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_true, pred_honest, target_names=["NEG", "POS"]))
    f.write(f"\nNeutral predictions: {total_neutral} ({total_neutral/len(y_true)*100:.2f}%)\n")
    f.write(f"  True Negative → Neutral: {neutral_on_neg}\n")
    f.write(f"  True Positive → Neutral: {neutral_on_pos}\n")

print(f"\nResults saved to: {RESULTS_PATH}/banglabook_zero_shot.txt")

Map:   0%|          | 0/30253 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
